# Representation-to-Image Decoder Training

This notebook loads a pretrained checkpoint (e.g., `resnet18`, `resnet18_mixer`, `split_resnet`, or `split_resnet_mixer`), freezes the encoder, and trains a decoder to reconstruct images from the encoder representation.

> **Important:** The decoder is trained on the **full dataset** (training + testing splits combined), not only the training split.

In [ ]:
from __future__ import annotations

import copy
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader
from omegaconf import OmegaConf

from visgen.datasets import Cars3D, CLEVR, DSprites, IRAVEN, MPI3D, Shapes3D
from visgen.models import get_model

plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
# ============ User configuration ============
# Preferred: point to the experiment cfg saved during training.
RUN_CFG_PATH = None  # e.g., "outputs/.../cfg.yml"

# Fallback if RUN_CFG_PATH is None:
BASE_CFG_PATH = "configs/base.yml"
DATA_CFG_PATH = "configs/datasets/dsprites.yml"
MODEL_CFG_PATH = "configs/models/resnet18.yml"
EXPERIMENT_CFG_PATH = "configs/experiments/iid.yml"

# Checkpoint from this repository's training code
CHECKPOINT_PATH = "/path/to/checkpoint_or_model_best.pth.tar"

# Decoder training hyperparameters
SEED = 7
BATCH_SIZE = 256
NUM_WORKERS = 2
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-6

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
def set_seed(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_cfg(
    run_cfg_path: str | None,
    base_cfg_path: str,
    data_cfg_path: str,
    model_cfg_path: str,
    experiment_cfg_path: str,
):
    if run_cfg_path is not None:
        cfg = OmegaConf.load(run_cfg_path)
        print(f"Loaded run config: {run_cfg_path}")
        return cfg

    cfg = OmegaConf.merge(
        OmegaConf.load(base_cfg_path),
        OmegaConf.load(model_cfg_path),
        OmegaConf.load(data_cfg_path),
        OmegaConf.load(experiment_cfg_path),
    )
    cfg["device"] = DEVICE
    print("Loaded merged cfg from base/data/model/experiment YAML files.")
    return cfg


def _to_dict(cfg_node):
    return OmegaConf.to_container(cfg_node, resolve=True)


def build_full_dataset(data_cfg):
    dataset_map = {
        "dsprites": DSprites,
        "iraven": IRAVEN,
        "mpi3d": MPI3D,
        "shapes3d": Shapes3D,
        "cars3d": Cars3D,
        "clevr": CLEVR,
    }

    tr_cfg = _to_dict(data_cfg.training)
    te_cfg = _to_dict(data_cfg.testing)

    dataset_name = tr_cfg["dataset"]
    dataset_cls = dataset_map[dataset_name]

    train_dataset = dataset_cls(**tr_cfg)
    test_dataset = dataset_cls(**te_cfg)

    full_dataset = ConcatDataset([train_dataset, test_dataset])

    print(f"Dataset: {dataset_name}")
    print(f"  train split size: {len(train_dataset)}")
    print(f"  test split size:  {len(test_dataset)}")
    print(f"  full size used for decoder training: {len(full_dataset)}")

    return full_dataset


def load_checkpoint_flexible(model: nn.Module, checkpoint_path: str, device: str):
    ckpt = torch.load(checkpoint_path, map_location=device)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
        print("Loaded `model_state_dict` from trainer checkpoint format.")
    elif isinstance(ckpt, dict):
        state_dict = ckpt
        print("Loaded checkpoint as a raw state_dict dictionary.")
    else:
        raise TypeError("Unsupported checkpoint format.")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Missing keys: {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")
    if len(missing) > 0:
        print("First missing keys:", missing[:10])
    if len(unexpected) > 0:
        print("First unexpected keys:", unexpected[:10])

    return model


def ensure_nchw(x: torch.Tensor) -> torch.Tensor:
    # Some models/trainers may hand over [B, V, C, H, W]; use the last view.
    if x.dim() == 5:
        x = x[:, -1]
    return x

In [ ]:
class RepresentationDecoder(nn.Module):
    """Simple MLP decoder from representation vector -> image tensor."""

    def __init__(self, rep_dim: int, image_shape: tuple[int, int, int], hidden_dim: int = 2048):
        super().__init__()
        c, h, w = image_shape
        out_dim = c * h * w
        self.image_shape = image_shape
        self.net = nn.Sequential(
            nn.Linear(rep_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.net(z)
        c, h, w = self.image_shape
        return x.view(z.shape[0], c, h, w)

In [ ]:
set_seed(SEED)

cfg = load_cfg(
    RUN_CFG_PATH,
    BASE_CFG_PATH,
    DATA_CFG_PATH,
    MODEL_CFG_PATH,
    EXPERIMENT_CFG_PATH,
)

model = get_model(cfg).to(DEVICE)
model = load_checkpoint_flexible(model, CHECKPOINT_PATH, DEVICE)
model.eval()

for p in model.parameters():
    p.requires_grad = False

full_dataset = build_full_dataset(cfg.data)

loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

# infer representation and image dimensions
sample_x, _ = full_dataset[0]
sample_x = ensure_nchw(sample_x.unsqueeze(0)).to(DEVICE)
with torch.no_grad():
    sample_z = model.extract_representation(sample_x)

rep_dim = int(sample_z.shape[-1])
image_shape = tuple(sample_x.shape[1:])

print(f"Representation dim: {rep_dim}")
print(f"Image shape: {image_shape}")

decoder = RepresentationDecoder(rep_dim=rep_dim, image_shape=image_shape).to(DEVICE)
optimizer = torch.optim.Adam(decoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print(decoder)

In [ ]:
loss_history = []

for epoch in range(1, EPOCHS + 1):
    decoder.train()
    epoch_loss = 0.0
    num_items = 0

    for x, _ in loader:
        x = ensure_nchw(x).to(DEVICE, non_blocking=True).float()

        with torch.no_grad():
            z = model.extract_representation(x)

        x_hat = decoder(z)
        loss = F.mse_loss(x_hat, x)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_size = x.shape[0]
        epoch_loss += loss.item() * batch_size
        num_items += batch_size

    epoch_loss /= max(num_items, 1)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch:03d}/{EPOCHS:03d} - MSE: {epoch_loss:.6f}")

In [ ]:
plt.figure()
plt.plot(loss_history)
plt.title("Decoder training loss (full dataset)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(True)
plt.show()

In [ ]:
@torch.no_grad()
def show_reconstructions(model, decoder, dataset, n=8):
    model.eval()
    decoder.eval()

    idxs = torch.randperm(len(dataset))[:n].tolist()
    xs = []
    for i in idxs:
        x, _ = dataset[i]
        x = ensure_nchw(x.unsqueeze(0)).squeeze(0)
        xs.append(x)

    x = torch.stack(xs, dim=0).to(DEVICE).float()
    z = model.extract_representation(x)
    x_hat = decoder(z).cpu()
    x = x.cpu()

    c = x.shape[1]
    fig, axes = plt.subplots(2, n, figsize=(2*n, 4))
    for i in range(n):
        xi = x[i].permute(1, 2, 0).numpy() if c > 1 else x[i, 0].numpy()
        xhi = x_hat[i].permute(1, 2, 0).numpy() if c > 1 else x_hat[i, 0].numpy()

        if c > 1:
            axes[0, i].imshow(xi.clip(0, 1))
            axes[1, i].imshow(xhi.clip(0, 1))
        else:
            axes[0, i].imshow(xi, cmap="gray", vmin=0.0, vmax=1.0)
            axes[1, i].imshow(xhi, cmap="gray", vmin=0.0, vmax=1.0)

        axes[0, i].axis("off")
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Original")
    axes[1, 0].set_ylabel("Reconstruction")
    plt.tight_layout()
    plt.show()

show_reconstructions(model, decoder, full_dataset, n=8)

In [ ]:
# Optional: save decoder weights
OUTPUT_DECODER_PATH = "decoder_from_representation.pt"
torch.save(
    {
        "decoder_state_dict": decoder.state_dict(),
        "rep_dim": rep_dim,
        "image_shape": image_shape,
        "epochs": EPOCHS,
        "loss_history": loss_history,
    },
    OUTPUT_DECODER_PATH,
)
print(f"Saved decoder to: {OUTPUT_DECODER_PATH}")